# COVID-19 Big Data Analytics — Exploration Notebook

**Project:** Distributed COVID-19 Big Data Analytics Platform  
**Author:** Aayush Savaliya (Member 3 — Analytics & NoSQL Engineer)  
**Dataset:** Google COVID-19 Open Data  

---
This notebook loads analytics results from MongoDB and visualizes:
- Global case & death trends (daily)
- Top 10 countries by total confirmed cases
- Vaccination progress by country
- Hospitalization burden


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import plotly.express as px
import plotly.graph_objects as go
from pymongo import MongoClient

# Connect to MongoDB
client = MongoClient('mongodb://localhost:27017')
db = client['covid_analytics']
print('Connected to MongoDB:', db.list_collection_names())

In [ ]:
# Load collections into pandas DataFrames
country_df = pd.DataFrame(list(db['country_summary'].find({}, {'_id': 0, '_loaded_at': 0})))
daily_df   = pd.DataFrame(list(db['daily_summary'].find({}, {'_id': 0, '_loaded_at': 0})))
vacc_df    = pd.DataFrame(list(db['vaccination_summary'].find({}, {'_id': 0, '_loaded_at': 0})))
hosp_df    = pd.DataFrame(list(db['hospitalization_summary'].find({}, {'_id': 0, '_loaded_at': 0})))

daily_df['date'] = pd.to_datetime(daily_df['date'])
daily_df = daily_df.sort_values('date')

print(f'Countries: {len(country_df)}')
print(f'Daily records: {len(daily_df)}')
print(f'Vaccination records: {len(vacc_df)}')
country_df.head()

In [ ]:
# ── Plot 1: Global Daily New Cases & 7-Day Rolling Average ────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(daily_df['date'], daily_df['global_new_confirmed'],
       color='steelblue', alpha=0.4, label='Daily New Cases')
ax.plot(daily_df['date'], daily_df['rolling_avg_confirmed_7d'],
        color='royalblue', linewidth=2, label='7-Day Rolling Avg')
ax.set_title('Global Daily New COVID-19 Cases', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('New Cases')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45)
ax.legend()
plt.tight_layout()
plt.savefig('../output/global_daily_cases.png', dpi=150)
plt.show()

In [ ]:
# ── Plot 2: Top 10 Countries by Total Confirmed Cases ────────────────────────
top10 = country_df.nlargest(10, 'total_confirmed')
fig = px.bar(
    top10, x='country_name', y='total_confirmed',
    color='case_fatality_rate', color_continuous_scale='Reds',
    title='Top 10 Countries — Total Confirmed Cases',
    labels={'total_confirmed': 'Total Confirmed', 'country_name': 'Country',
            'case_fatality_rate': 'CFR (%)'}
)
fig.write_html('../output/top10_countries.html')
fig.show()

In [ ]:
# ── Plot 3: Vaccination Progress — Top 15 Countries ──────────────────────────
top15_vacc = vacc_df.nlargest(15, 'total_vaccinated')
fig = px.bar(
    top15_vacc, x='total_vaccinated', y='country_name',
    orientation='h',
    title='Top 15 Countries — Total Vaccinations',
    labels={'total_vaccinated': 'Total Vaccinated', 'country_name': 'Country'},
    color='total_vaccinated', color_continuous_scale='Greens'
)
fig.write_html('../output/vaccination_progress.html')
fig.show()

In [ ]:
# ── Plot 4: Case Fatality Rate by Country (Top 20) ───────────────────────────
top20_cfr = country_df[country_df['total_confirmed'] > 10000].nlargest(20, 'case_fatality_rate')
fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(top20_cfr['country_name'], top20_cfr['case_fatality_rate'],
               color='tomato', edgecolor='white')
ax.set_xlabel('Case Fatality Rate (%)')
ax.set_title('Countries with Highest Case Fatality Rate\n(min. 10,000 confirmed cases)', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../output/case_fatality_rate.png', dpi=150)
plt.show()

client.close()
print('Analysis complete. Output saved to /output/')